# Mental Health Sentiment Analysis - Model Training (Google Colab)

This notebook trains the **Hybrid Lexicon-RoBERTa Classifier** for sentiment and psychological analysis of social media text. It extracts features using a pre-trained RoBERTa model (`cardiffnlp/twitter-roberta-base-sentiment-latest`) and the NRC Emotion Lexicon, combines them, and trains a PyTorch classification network.

## 🛠️ Step 1: Install Dependencies
We need to install Hugging Face's `transformers` library, `nrclex` for lexicon features, and other standard libraries.

In [ ]:
!pip install transformers torch pandas scikit-learn nrclex

## 📂 Step 2: Upload or Mount Google Drive
Run this cell to mount your Google Drive if you have uploaded your dataset there, or you can use the Colab file upload panel on the left to upload your `mental_health_posts.csv` dataset.

In [ ]:
from google.colab import drive
# drive.mount('/content/drive')  # Uncomment if you want to load from Drive

## 🧹 Step 3: Text Preprocessing
This defines the text cleaning and label mappings, matching the logic in the project's codebase.

In [ ]:
import re
import pandas as pd
from typing import Tuple

EMOTION_LABELS = ["Happy/Positive", "Neutral", "Anxious/Stress", "Depressed/Sad"]
EMOTION_LABEL2ID = {label: idx for idx, label in enumerate(EMOTION_LABELS)}
EMOTION_ID2LABEL = {idx: label for label, idx in EMOTION_LABEL2ID.items()}

SENTIMENT_LABELS = ["Negative", "Neutral", "Positive"]
SENTIMENT_LABEL2ID = {label: idx for idx, label in enumerate(SENTIMENT_LABELS)}

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags symbol but keep text
    text = re.sub(r'#(\w+)', r'\1', text)
    # Compress repeated punctuation
    text = re.sub(r'([!?.,])\1+', r'\1', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_and_prepare_labeled_dataset(
    file_path: str,
    text_column: str = "full_text",
    emotion_column: str = "emotion_label",
    sentiment_column: str = "sentiment",
    min_words: int = 3,
) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    df = df.dropna(subset=[text_column, emotion_column, sentiment_column]).copy()
    df['cleaned_text'] = df[text_column].apply(clean_text)
    df = df.drop_duplicates(subset=['cleaned_text'])
    df = df[df['cleaned_text'].str.split().str.len() >= min_words]
    df = df[df[emotion_column].isin(EMOTION_LABELS)]
    df = df[df[sentiment_column].isin(SENTIMENT_LABELS)]
    df['emotion_label_id'] = df[emotion_column].map(EMOTION_LABEL2ID)
    df['sentiment_label_id'] = df[sentiment_column].map(SENTIMENT_LABEL2ID)
    return df.reset_index(drop=True)

## 🧠 Step 4: Load and Test a Pre-trained RoBERTa Embeddings Model
We load `cardiffnlp/twitter-roberta-base-sentiment-latest` to compute embeddings.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
roberta_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
roberta_model.eval()
print("RoBERTa Model loaded successfully.")

## 🏷️ Step 5: Feature Extraction Functions (RoBERTa & NRC Lexicon)
We extract: 
1. **RoBERTa contextual embedding** (768 dimensions), utilizing mask-weighted mean pooling to capture the context.
2. **NRC Lexicon frequencies** (10 dimensions) indicating raw emotion word frequencies.

In [ ]:
import numpy as np
from nrclex import NRCLex
from tqdm import tqdm

NRC_EMOTIONS = ["fear", "anger", "anticipation", "trust", "surprise",
                "positive", "negative", "sadness", "disgust", "joy"]

def extract_nrc_features(text: str) -> list:
    if not isinstance(text, str) or not text.strip():
        return [0.0] * len(NRC_EMOTIONS)
    lex = NRCLex()
    lex.load_raw_text(text)
    scores = lex.affect_frequencies
    return [float(scores.get(emotion, 0.0)) for emotion in NRC_EMOTIONS]

@torch.no_grad()
def get_roberta_embeddings_batch(texts: list, batch_size: int = 32, max_length: int = 256) -> np.ndarray:
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting RoBERTa Embeddings"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        out = roberta_model(**enc)
        last_hidden = out.last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1).float()
        # Mean pool weighted by attention mask
        pooled = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        all_embeddings.append(pooled.cpu().numpy())
    return np.vstack(all_embeddings)

## 📈 Step 6: Load and Preprocess Your Dataset
**Note:** Ensure you upload the CSV dataset file to Colab or Google Drive and specify the path here. We'll load, preprocess, and split the data.

In [ ]:
# Change this path to match your CSV file location (e.g. "mental_health_posts.csv")
CSV_PATH = "mental_health_posts.csv" 

# For demonstration purposes, if the CSV is not found, we create a tiny mock dataset so the notebook is runnable
import os
if not os.path.exists(CSV_PATH):
    print(f"Dataset '{CSV_PATH}' not found! Creating a small dummy dataset for testing.")
    dummy_data = {
        "full_text": [
            "I feel so happy today, everything is bright and cheerful!",
            "Just a standard daily routine post with regular stuff.",
            "I cannot breathe, my heart is pounding, and I feel so stressed out and anxious.",
            "I feel completely hopeless and tired of everything. I just want to lie in bed.",
            "What a great day for a walk in the park!",
            "The weather is average, nothing special happening.",
            "I am overwhelmed with pressure from work and school, extremely stressed.",
            "Deeply sad and lonely tonight, crying alone in my room."
        ] * 10,  # Multiply to have 80 rows
        "emotion_label": [
            "Happy/Positive", "Neutral", "Anxious/Stress", "Depressed/Sad",
            "Happy/Positive", "Neutral", "Anxious/Stress", "Depressed/Sad"
        ] * 10,
        "sentiment": [
            "Positive", "Neutral", "Negative", "Negative",
            "Positive", "Neutral", "Negative", "Negative"
        ] * 10
    }
    pd.DataFrame(dummy_data).to_csv(CSV_PATH, index=False)

df = load_and_prepare_labeled_dataset(CSV_PATH)
print(f"Loaded dataset. Size: {len(df)} rows")
print("Emotion Label distribution:\n", df['emotion_label'].value_counts())

## 🏷️ Step 7: Pre-Extract Features to Speed up Training
Because RoBERTa is kept frozen during hybrid model training, we can extract the RoBERTa embeddings and Lexicon vectors for all texts once before training. This speeds up training by 100x!

In [ ]:
print("Extracting RoBERTa Embeddings (this may take a few minutes if the dataset is large)...")
texts = df['cleaned_text'].tolist()
roberta_feats = get_roberta_embeddings_batch(texts, batch_size=32)

print("Extracting NRC Lexicon Features...")
nrc_feats = np.array([extract_nrc_features(t) for t in tqdm(texts, desc="Extracting Lexicon")])

# Combine features into a single array
X_features = np.hstack([roberta_feats, nrc_feats])
y_labels = df['emotion_label_id'].values

print(f"Feature matrix shape: {X_features.shape}")
print(f"Label vector shape: {y_labels.shape}")

## 📊 Step 8: Train-Validation-Test Split
Split our extracted features using a stratified split (80% train, 10% val, 10% test).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train size: {X_train.shape[0]}, Val size: {X_val.shape[0]}, Test size: {X_test.shape[0]}")

## 🏛️ Step 9: Define the Hybrid Classification Network
This matches the architecture of `HybridSentimentClassifier` in the Flask project codebase.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class HybridSentimentClassifier(nn.Module):
    def __init__(self, roberta_dim: int = 768, lexicon_dim: int = 10, num_labels: int = 4):
        super(HybridSentimentClassifier, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(roberta_dim + lexicon_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_labels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)

# Initialize model
model = HybridSentimentClassifier(roberta_dim=768, lexicon_dim=10, num_labels=4).to(device)
print(model)

## 🔄 Step 10: PyTorch Dataset & Training Loop
Set up training parameters, batching, and train the model while evaluating on the validation set after each epoch.

In [ ]:
# Convert data to PyTorch Tensors
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Define loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Training settings
epochs = 20
best_val_loss = float('inf')
save_path = "best_hybrid_model.pt"

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_x.size(0)
        _, predicted = outputs.max(1)
        total_train += batch_y.size(0)
        correct_train += predicted.eq(batch_y).sum().item()
        
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_x.size(0)
            _, predicted = outputs.max(1)
            total_val += batch_y.size(0)
            correct_val += predicted.eq(batch_y).sum().item()
            
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = correct_train / total_train
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
    
    # Save best checkpoint
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), save_path)
        print(f"  --> Saved new best checkpoint to '{save_path}'")

## 🧪 Step 11: Evaluate on the Test Set
We load the best model weights and calculate validation metrics (Accuracy, Precision, Recall, F1-Score) and a classification report.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Load best weights
model.load_state_dict(torch.load(save_path))
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.numpy())

print("Test Accuracy:", accuracy_score(all_targets, all_preds))
print("\nClassification Report:")
print(classification_report(all_targets, all_preds, target_names=EMOTION_LABELS))

## 📥 Step 12: Download the Trained Model weights
After training, you need to download `best_hybrid_model.pt` and place it in your local directory at `models/roberta_finetuned/best_hybrid_model.pt` so the web app can use it.

In [ ]:
from google.colab import files
try:
    files.download('best_hybrid_model.pt')
    print("Triggered download of 'best_hybrid_model.pt'")
except Exception as e:
    print("Download trigger failed (might be running in standard non-Colab Jupyter). Locate and download the file manually:", e)